In [47]:
# Packages
import os
import re

# For downloading NOAA data
import gzip
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
import time
import requests

# For data analysis
import pandas as pd
import numpy as np


In [ ]:
# Links of gzip storm data from NOAA website
# https://www.ncei.noaa.gov/stormevents/ftp.jsp

def storm_data():
    # Web scrape NOAA weather data links
    storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
    req = requests.get(storms_url)                   # access url webpage
    soup = BeautifulSoup(req.text, 'html.parser')    # parse thru HTML text of webpage

    # Find 2000s data.csv.gz filenames in <a href="link" > format 
    pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'    # 2000s filename pattern
    data_links = []
    for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):
        year_data = link.get('href')
        full_link = str(storms_url) + str(year_data)
        data_links.append(full_link)
    return data_links


# Parallel download func for data
def download_files(data):
    # Create new directory for data
    data_dir = '../data'
    os.makedirs(data_dir, exist_ok=True)
    
    # Check url request for 'content-disposition' header to parse .gz filenames
    response = requests.get(data, stream=True)
    if 'content-disposition' in response.headers:
        content_disp = response.headers['content-disposition']
        file_name = content_disp.split('filename=')[1]
    else:
        file_name = data.split('/')[-1]
    
    # Write downloaded gzip data to data dir
    gz_name = os.path.join(data_dir, file_name)
    with open(gz_name, 'wb') as gz_file:
        gz_file.write(response.content)
    print(f'Downloaded file to {gz_name}')


# Use ThreadPoolExecutor() to parallel download gzip files
with ThreadPoolExecutor() as executor:
    executor.map(download_files, storm_data())

Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2002_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2006_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2001_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2005_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2000_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2004_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2009_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2003_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2007_c20260323.csv.gz
Downloaded file to ../data/StormEvents_details-ftp_v

In [ ]:
# Unzip gzip file into csv format & read using pandas
df = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz', compression='gzip', header=0)
df['EVENT_TYPE'].unique()
'''
weather interested in
flash flood, tornado, excess heat, winter weather, wildfire, winter storm, extreme cold/wind chill,
heavy snow, drought, heat, frost/freeze, lake-effect snow, hurricane (Typhoon), 
sleet, flood, ice storm, hail, blizzard, funnel cloud, freezing fog, tropical storm,
storm surge/tide, tropical depression, cold/wind chill, lightning, thunderstorm wind
'''


<StringArray>
[               'Flash Flood',                      'Flood',
                    'Tornado',                  'Ice Storm',
             'Excessive Heat',                       'Hail',
             'Winter Weather',          'Thunderstorm Wind',
                   'Wildfire',                'Debris Flow',
               'Winter Storm',                 'Heavy Rain',
    'Extreme Cold/Wind Chill',                  'Lightning',
                 'Heavy Snow',                  'High Wind',
                'Strong Wind',   'Marine Thunderstorm Wind',
                'Rip Current',            'Cold/Wind Chill',
                 'Waterspout',                  'High Surf',
                'Marine Hail',                   'Blizzard',
                    'Drought',                  'Dense Fog',
                       'Heat',               'Funnel Cloud',
               'Frost/Freeze',                  'Avalanche',
              'Coastal Flood',           'Marine High Wind',
      'Ast